# EX — Software Engineering Tooling Real-World Exercises

REST API design, an LRU cache (Redis-style, in-memory), a pub-sub queue simulation
(Kafka-style), core design patterns, heaps/tries, and regex for data extraction.


## 1. REST API Design Principles
**Pointer:** design endpoints around resources ('/orders/{id}'), not actions ('/getOrder').

In [ ]:
# A minimal in-memory REST-style resource handler (no server needed -- just the design pattern)
class OrdersAPI:
    def __init__(self):
        self.orders = {1: {"id":1,"status":"shipped"}, 2: {"id":2,"status":"pending"}}
        self._next_id = 3

    def get(self, order_id):
        order = self.orders.get(order_id)
        return (200, order) if order else (404, {"error":"not found"})

    def list(self):
        return (200, list(self.orders.values()))

    def create(self, data):
        order_id = self._next_id
        self._next_id += 1
        self.orders[order_id] = {"id": order_id, **data}
        return (201, self.orders[order_id])

api = OrdersAPI()
print(api.get(1))
print(api.create({"status":"pending"}))
print(api.get(99))


### TODO 1
Add `update(order_id, data)` (returns 200 + updated resource, or 404) and `delete(order_id)` (returns 204 with no body, or 404) methods, following the same status-code conventions.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
def update(self, order_id, data):
    if order_id not in self.orders:
        return (404, {"error": "not found"})
    self.orders[order_id].update(data)
    return (200, self.orders[order_id])

def delete(self, order_id):
    if order_id not in self.orders:
        return (404, {"error": "not found"})
    del self.orders[order_id]
    return (204, None)

OrdersAPI.update = update
OrdersAPI.delete = delete
```
</details>


## 2. LRU Cache (Redis-style, in-memory)
**Pointer:** an LRU cache turns 'expensive to recompute, same input repeated often' into a fast lookup.

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity=3):
        self.capacity = capacity
        self.store = OrderedDict()

    def get(self, key):
        if key not in self.store:
            return None
        self.store.move_to_end(key)  # mark as recently used
        return self.store[key]

    def put(self, key, value):
        if key in self.store:
            self.store.move_to_end(key)
        self.store[key] = value
        if len(self.store) > self.capacity:
            self.store.popitem(last=False)  # evict least-recently-used

cache = LRUCache(capacity=2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")       # 'a' is now most-recently-used
cache.put("c", 3)    # should evict 'b' (least recently used), not 'a'
print(list(cache.store.keys()))


### TODO 2
Wrap an expensive function with a caching decorator built on `LRUCache`, and demonstrate a repeated call is much faster on a cache hit (compare timing).

In [ ]:
import time
# TODO
def expensive_call(n):
    time.sleep(0.05)
    return n ** 2

def cached(fn, capacity=3):
    pass

fast_expensive = cached(expensive_call) if 'cached' in dir() else expensive_call


<details><summary>Solution</summary>

```python
def cached(fn, capacity=3):
    cache = LRUCache(capacity)
    def wrapper(n):
        hit = cache.get(n)
        if hit is not None:
            return hit
        result = fn(n)
        cache.put(n, result)
        return result
    return wrapper

fast_expensive = cached(expensive_call)
t0 = time.time(); fast_expensive(5); print("first call:", time.time()-t0)
t0 = time.time(); fast_expensive(5); print("cached call:", time.time()-t0)
```
</details>


## 3. Pub-Sub / Message Queue Simulation (Kafka-style)
**Pointer:** the point of a queue is decoupling — the producer doesn't need the consumer online right now.

In [ ]:
from collections import deque

class SimpleQueue:
    def __init__(self):
        self.topics = {}
    def publish(self, topic, message):
        self.topics.setdefault(topic, deque()).append(message)
    def consume(self, topic):
        q = self.topics.get(topic, deque())
        return q.popleft() if q else None

mq = SimpleQueue()
mq.publish("orders.created", {"order_id": 1})
mq.publish("orders.created", {"order_id": 2})
print(mq.consume("orders.created"))
print(mq.consume("orders.created"))
print(mq.consume("orders.created"))  # None, queue empty


### TODO 3
Extend `SimpleQueue` to support multiple named consumers per topic, each tracking their own read position independently (like Kafka consumer groups) — so two different consumers can each read all messages without interfering with each other.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
class ConsumerGroupQueue:
    def __init__(self):
        self.topics = {}       # topic -> list of messages
        self.offsets = {}      # (topic, consumer_name) -> next index to read

    def publish(self, topic, message):
        self.topics.setdefault(topic, []).append(message)

    def consume(self, topic, consumer_name):
        messages = self.topics.get(topic, [])
        offset = self.offsets.get((topic, consumer_name), 0)
        if offset >= len(messages):
            return None
        self.offsets[(topic, consumer_name)] = offset + 1
        return messages[offset]
```
</details>


## 4. Design Patterns — Singleton, Factory, Observer

In [ ]:
# Singleton -- exactly one instance shared everywhere (e.g. a config object)
class Config:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.settings = {}
        return cls._instance

c1, c2 = Config(), Config()
c1.settings["debug"] = True
print(c2.settings)  # same object -- shows True

# Factory -- centralize object creation logic
class ModelFactory:
    @staticmethod
    def create(model_type):
        if model_type == "classifier": return {"type":"classifier", "layers":3}
        if model_type == "regressor": return {"type":"regressor", "layers":2}
        raise ValueError(f"unknown type {model_type}")

print(ModelFactory.create("classifier"))


### TODO 4
Implement the Observer pattern: a `EventBus` class with `subscribe(event, callback)` and `publish(event, data)` that calls all subscribed callbacks for that event.

In [ ]:
# TODO
class EventBus:
    pass

bus = EventBus() if 'EventBus' in dir() and EventBus.__init__ != object.__init__ else None


<details><summary>Solution</summary>

```python
class EventBus:
    def __init__(self):
        self.subscribers = {}
    def subscribe(self, event, callback):
        self.subscribers.setdefault(event, []).append(callback)
    def publish(self, event, data):
        for cb in self.subscribers.get(event, []):
            cb(data)

bus = EventBus()
bus.subscribe("order.created", lambda d: print("send confirmation email for", d))
bus.subscribe("order.created", lambda d: print("update inventory for", d))
bus.publish("order.created", {"order_id": 1})
```
</details>


## 5. Efficient Data Structures — Heap & Trie

In [ ]:
import heapq

# Heap: always process the highest-priority support ticket next
tickets = []
heapq.heappush(tickets, (2, "slow app"))       # (priority, description) -- lower number = higher priority
heapq.heappush(tickets, (1, "site is down"))
heapq.heappush(tickets, (3, "minor UI bug"))

while tickets:
    priority, desc = heapq.heappop(tickets)
    print(priority, desc)


### TODO 5
Implement a minimal Trie (prefix tree) with `insert(word)` and `starts_with(prefix) -> bool`, useful for autocomplete.

In [ ]:
# TODO
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
    def insert(self, word):
        pass
    def starts_with(self, prefix):
        pass

trie = Trie()
for w in ["cat", "car", "dog"]:
    trie.insert(w)
print(trie.starts_with("ca"))   # True
print(trie.starts_with("do"))   # True
print(trie.starts_with("z"))    # False


<details><summary>Solution</summary>

```python
def insert(self, word):
    node = self.root
    for ch in word:
        node = node.children.setdefault(ch, TrieNode())
    node.is_end = True

def starts_with(self, prefix):
    node = self.root
    for ch in prefix:
        if ch not in node.children:
            return False
        node = node.children[ch]
    return True

Trie.insert = insert
Trie.starts_with = starts_with
```
</details>


## 6. Regex for Data Extraction
**Pointer:** test regex against messy, real-world edge cases, not just the happy path.

In [ ]:
import re

text = '''
Contact john.doe@example.com or call (555) 123-4567.
Invoice total: $1,234.56. Also reach sales@company.co at +1-800-555-0100.
'''

emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
prices = re.findall(r"\$[\d,]+\.\d{2}", text)
print(emails)
print(prices)


### TODO 6
Write a regex to extract all phone numbers in the text (handle both `(555) 123-4567` and `+1-800-555-0100` formats).

In [ ]:
# TODO
phone_pattern = r""
phones = re.findall(phone_pattern, text)
print(phones)


<details><summary>Solution</summary>

```python
phone_pattern = r"(?:\+\d{1,2}-?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}"
phones = re.findall(phone_pattern, text)
```
</details>


## Key Takeaways
- REST endpoints model resources with standard HTTP verbs/status codes, not custom action names.
- An LRU cache (OrderedDict-based) is the in-memory analog of what Redis provides as a service.
- Message queues decouple producers and consumers in time — this is the entire point of Kafka/RabbitMQ-style systems.
- Reach for design patterns when you feel the specific pain they solve, not by default.
- Heaps give O(log n) priority processing; tries give fast prefix lookups (autocomplete).
- Test regex against messy real-world input, not just the clean example.
